<div align="right"><sub>Notebook 最終更新: 2026-04-23 09:15</sub></div>
<h1><strong>05. LLMエージェントの基礎</strong></h1>

今回からは、LLMを単体で使うのではなく、役割を持った「エージェント」として組み合わせて、複雑なタスクをこなす方法を学びます。

**Qwen3-8B のような現代の強力なモデルは「指示漏れ」や「フォーマット違反」のような単純なミスをほとんどしません。** 
そこでこの回では、エージェントの目的を「間違い探し」から**「さらなる品質向上（壁打ちによる品質向上）」**へとステップアップさせます。

回答を行う **Writer（執筆者）** と、それをレビューして改善要求を出す **Editor（編集長）** の2役を組み合わせたループを体験しましょう。

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

from src.common import load_llm, generate_text, AGENT_MODEL_ID
from src.agent_core import LLMExecutorCriticAgent, RoleConfig

model, tokenizer = load_llm(model_id=AGENT_MODEL_ID)
print('準備完了')


## **1. チャット関数の準備**

In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 768, temp: float = 0.5):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

## **2. 執筆者と編集長のエージェント構築**
Writer（Executor）が書いた初稿に対して、Editor（Critic）が必ず「具体例の追加」と「トーンの変更」を要求し、Writerがそれに答えて書き直すプロセスを実行します。

In [ ]:
writer_prompt = """
あなたはプロのライターです。ユーザーからのテーマについて記事を書いてください。
Editorから修正指示が来た場合は、その内容を必ず反映し、前回の文章から明確に改善された文章を書き直してください。
同じ内容の再提出は禁止です。必ず具体的な表現や描写を追加・変更してください。
"""

editor_prompt = """
あなたは非常に厳しい編集長です。提出された文章を読み、以下の2つの基準が【両方とも】十分な水準で満たされているか評価してください。
1つ目は、読者が日常生活の中で具体的にイメージできる活用シーンが含まれていることです。ただし、「通勤」などの単語があるだけでは不十分であり、時間・場所・行動が伴った情景として描写されているかを確認してください。
2つ目は、読者が体験してみたいと感じるような、ワクワクする感情豊かなトーンになっていることです。ただし、「希望」「可能性」などの抽象的な表現だけに依存している場合は不十分であり、具体的な体験と結びついた表現になっているかを確認してください。
さらに、比喩や抽象的な感情表現だけでなく、実際の感覚や行動の流れが描かれているかも確認してください。
各基準について必ず「満たしている」または「満たしていない」のどちらかを明確に判定してください。
もし、上記のうち1つでも満たしていない場合は、不足している点を具体的に指摘して書き直しを要求してください。
具体的な時間・場所・行動が明確に含まれ、かつ少なくとも1つの感覚的な体験（視覚・聴覚・触覚など）が描写されている場合は、十分と判断してください。
文章全体として、テーマについての解説（仕組みや特徴の説明）が含まれているかも確認してください。
【重要】絶対に自分で文章を書き直さず、Writerへの修正指示だけを簡潔に出力してください。評価の途中説明や同じ指摘の繰り返しは禁止します。最終判断のみを出してください。
すべての条件が十分に満たされていると判断した場合のみ、「誤りなし」と出力してください。
"""

writer = RoleConfig(name="Writer", system_prompt=writer_prompt)
editor = RoleConfig(name="Editor", system_prompt=editor_prompt)

agent = LLMExecutorCriticAgent(llm_chat, role_configs=[writer, editor])

query = "未来の交通手段である「空飛ぶクルマ（eVTOL）」が普及した社会について、300字程度で解説記事を書いてください。"#@param{type:'string'}

# 最大3イテレーション（初稿発行 → 編集長指摘 → 第2稿発行 → 編集長指摘 → 第3稿発行）で実行します
final_answer, full_log, steps = agent.run_pipeline(query, max_iterations=3)

print("=== エージェントの処理過程 ===")
print(full_log)

print("\n=== ✅ 最終回答（第3稿） ===")
print(final_answer)

## **3. 何が起きたのか？（処理過程の分析）**

`=== エージェントの処理過程 ===` のログを読むと、以下の流れがはっきりと確認できます。

1. **Writer**: まず、テーマに沿った解説記事（初稿）を作成します。
2. **Editor**: その初稿を読み、システムプロンプトに定義された「3つの評価基準（解説の有無、活用シーンの具体性、感情豊かな抽象表現でないか）」に照らし合わせて厳密に評価し、不足している部分があれば的確に修正指示を出します。
3. **推敲ループ**: Writer はその指示を吸収して改稿し、Editor は再度基準に照らして評価します。この「執筆 → 評価・ダメ出し → 修正」のキャッチボール（協調動作）が、Editor が「すべての基準を十分に満たした（誤りなし）」と判定するまで繰り返されます。

1回のプロンプト（Zero-shot）で最初から完璧な構成や独自のトーンを引き出すのは至難の業ですが、このように**「書く役割」と「評価する役割」を明確に分担させる（Executor-Critic パターン）**ことで、LLMは客観的な視点を取り入れ、劇的に出力の質を高めることができます。

---
### **まとめ**
- 現代の優秀なLLMを用いたエージェント構造は、単純な「ミスの修正」にとどまらず、**「要求水準に対する品質の引き上げ（壁打ちによる推敲）」**として絶大な効果を発揮します。
- 内部的には単一のLLMであっても、システムプロンプトの切り替えによって**「異なる役割（ライター、編集長、監査役など）」**を持たせて対話させることで、システム全体が多角的な視点を持つようになります。
- このようなエージェントに評価を行わせる際は、「前に何を言ったか」という記憶に頼るのではなく、**「何が達成されていれば合格とするか」という客観的で厳密なチェックリスト（評価基準）を与える**ことで、無限ループを防ぎ、確実で質の高い協調動作を実現できます。